In [ ]:
# Required libraries
import pandas as pd
from scipy.io import arff
from sklearn.feature_selection import f_classif
import matplotlib.pyplot as plt
import seaborn as sns

# Load the ARFF data
data, meta = arff.loadarff('diabetes.arff')

# Convert to pandas DataFrame
df = pd.DataFrame(data)

# Ensure the 'Outcome' column is treated as categorical (0, 1)
df['Outcome'] = df['Outcome'].astype(int)

# Separate features and target
X = df.drop('Outcome', axis=1)  # features
y = df['Outcome']               # target

# Apply ANOVA F-test
f_values, p_values = f_classif(X, y)

# Create a DataFrame to show the results
anova_results = pd.DataFrame({
    'Feature': X.columns,
    'F-Value': f_values,
    'P-Value': p_values
}).sort_values(by='F-Value', ascending=False)

# Display the features with the best and worst discriminative power
best_feature = anova_results.iloc[0]['Feature']
worst_feature = anova_results.iloc[-1]['Feature']

print(f"Best Feature: {best_feature}")
print(f"Worst Feature: {worst_feature}")

# Plot class-conditional probability density functions for best and worst features
plt.figure(figsize=(12, 6))

# Best Feature KDE plot
plt.subplot(1, 2, 1)
sns.kdeplot(df[best_feature][df['Outcome'] == 0], label='Class 0 (Normal)', fill=True)
sns.kdeplot(df[best_feature][df['Outcome'] == 1], label='Class 1 (Diabetes)', fill=True)
plt.title(f'Class-Conditional KDE for Best Feature: {best_feature}')
plt.xlabel(best_feature)
plt.legend()

# Worst Feature KDE plot
plt.subplot(1, 2, 2)
sns.kdeplot(df[worst_feature][df['Outcome'] == 0], label='Class 0 (Normal)', fill=True)
sns.kdeplot(df[worst_feature][df['Outcome'] == 1], label='Class 1 (Diabetes)', fill=True)
plt.title(f'Class-Conditional KDE for Worst Feature: {worst_feature}')
plt.xlabel(worst_feature)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Required libraries
import pandas as pd
from scipy.io import arff
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Load the ARFF data
data, meta = arff.loadarff('diabetes.arff')

# Convert to pandas DataFrame
df = pd.DataFrame(data)

# Ensure the 'Outcome' column is treated as categorical (0, 1)
df['Outcome'] = df['Outcome'].astype(int)

# Separate features and target
X = df.drop('Outcome', axis=1)  # features
y = df['Outcome']               # target

# Perform a stratified 80-20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

# Values for min_samples_split
min_samples_splits = [2, 5, 10, 20, 30, 50, 100]

# Lists to store accuracies
train_accuracies = []
test_accuracies = []

# Loop through the different min_samples_split values
for min_samples in min_samples_splits:
    # Initialize Decision Tree Classifier with given min_samples_split
    clf = DecisionTreeClassifier(min_samples_split=min_samples, random_state=1)
    
    # Train the model on training data
    clf.fit(X_train, y_train)
    
    # Predict on training and testing sets
    y_train_pred = clf.predict(X_train)
    y_test_pred = clf.predict(X_test)
    
    # Calculate accuracy for both training and testing sets
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    
    # Append accuracies to the respective lists
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(min_samples_splits, train_accuracies, label='Training Accuracy', marker='o')
plt.plot(min_samples_splits, test_accuracies, label='Testing Accuracy', marker='o')

# Add labels and title
plt.xlabel('min_samples_split')
plt.ylabel('Accuracy')
plt.title('Training and Testing Accuracies vs. min_samples_split')
plt.legend()

# Show the plot
plt.grid(True)
plt.show()


3) 

Key Observations:
Training Accuracy:

Decreases with higher min_samples_split values: As min_samples_split increases, the training accuracy declines. This is expected because increasing min_samples_split prevents the tree from growing too deep, reducing its capacity to fit the training data perfectly (i.e., reducing overfitting). At the highest value of min_samples_split (100), the tree is relatively shallow, so it struggles to capture more complex patterns in the training data.

Overfitting with small min_samples_split values: When min_samples_split is low (especially at 2), the training accuracy is very high (near 100%), indicating that the model has perfectly fit the training data, but this is often a sign of overfitting. The tree is likely memorizing the training data rather than learning generalizable patterns.

Testing Accuracy:

Relatively lower than training accuracy: The testing accuracy is consistently lower than the training accuracy across all values of min_samples_split. This is typical because testing on unseen data is harder, and it indicates that the model overfits the training data at lower min_samples_split values.

Generalization performance improves with moderate min_samples_split values: For low values of min_samples_split (e.g., 2, 5), the testing accuracy is quite low (around 0.70-0.72), suggesting that the model is overfitting. However, as min_samples_split increases (up to around 20 or 30), the testing accuracy improves and stabilizes at around 0.75. This indicates that the model is generalizing better and is less prone to overfitting with moderate values of min_samples_split.

Testing accuracy slightly decreases after 30: After a min_samples_split of 30, the testing accuracy begins to decline again, indicating that the model might be underfitting. This happens because the tree becomes too shallow and lacks the flexibility to capture important patterns in the data.

Sweet Spot Around min_samples_split = 20-30:

The plot suggests that the best generalization occurs when min_samples_split is around 20 or 30. At this point, the model achieves a balance between overfitting and underfitting, leading to reasonable training accuracy (around 85%) and testing accuracy (around 75%).
Generalization Capacity:
Overfitting at Low min_samples_split Values: When min_samples_split is set to low values like 2 or 5, the model overfits the training data. This is reflected by the near-perfect training accuracy but much lower testing accuracy. The tree is too deep and captures noise in the training set that doesn't generalize well to new, unseen data.

Better Generalization at Moderate Values (20-30): As min_samples_split increases, overfitting is mitigated, leading to better generalization. A decision tree with min_samples_split between 20 and 30 strikes a good balance between model complexity and generalization, as indicated by stable testing accuracy and reduced overfitting.

Underfitting at High min_samples_split Values: For values like 50 or 100, the testing accuracy drops slightly. This suggests the model is underfitting, i.e., it is too simple to capture the underlying patterns of the data due to restrictions on tree depth (caused by high min_samples_split), leading to poor performance on both training and testing data.

Conclusion:
Generalization: The model generalizes best when min_samples_split is set to a moderate value (around 20-30). At very low values, the model overfits, capturing noise from the training data, while at high values, the model underfits, failing to capture important patterns.

Practical Recommendation: Based on the graph, setting min_samples_split to around 20-30 would be ideal for achieving a balance between high accuracy and good generalization to unseen data in this specific dataset.

In [ ]:
# Import required libraries
import pandas as pd
from scipy.io import arff
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
import matplotlib.pyplot as plt

# Load the ARFF data
data, meta = arff.loadarff('diabetes.arff')

# Convert to pandas DataFrame
df = pd.DataFrame(data)

# Ensure the 'Outcome' column is treated as categorical (0, 1)
df['Outcome'] = df['Outcome'].astype(int)

# Separate features and target
X = df.drop('Outcome', axis=1)  # features
y = df['Outcome']               # target

# Train a Decision Tree Classifier on the entire dataset with max_depth=3
clf = DecisionTreeClassifier(max_depth=3, random_state=1)
clf.fit(X, y)

# Plot the decision tree
plt.figure(figsize=(20,10))
tree.plot_tree(clf, feature_names=X.columns, class_names=['Normal', 'Diabetes'], filled=True)
plt.show()
